# Stretch: more questions

For when the ladder or the joins exercise is done and there is time left.
These get harder as you go, so nobody is expected to finish them all.
1 to 3 use one table at a time; 4 to 8 need joins.

Before you run anything, say out loud roughly what you expect the answer to
be. Several of these are built to catch a query that runs fine and answers
the wrong question. After every join, check the row count.

A few need something new, and the notebook tells you what it is when you
get there.

In [ ]:
import os
import sqlite3
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "music.db").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

con = sqlite3.connect("data/music.db")


def run(sql, limit=10):
    """Run a query and print the rows as an aligned table."""
    cursor = con.execute(sql)
    headers = [d[0] for d in cursor.description]
    rows = cursor.fetchall()
    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows[:limit] or [[""]]))
              for i, h in enumerate(headers)]
    print("  ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print("  ".join("-" * w for w in widths))
    for row in rows[:limit]:
        print("  ".join(str(v).ljust(w) for v, w in zip(row, widths)))
    print(f"({len(rows)} rows)" if len(rows) <= limit
          else f"... {len(rows)} rows in total")

## Stretch 1: how many artists are still active?

`still_active` is `1` for yes and `0` for no. Here are three ways to ask
the question. Before running them, predict: do they all give the same
answer?

```sql
SELECT COUNT(still_active) FROM artists
SELECT SUM(still_active) FROM artists
SELECT COUNT(*) FROM artists WHERE still_active = 1
```

In [ ]:
run("""
""")

In [ ]:
run("""
""")

In [ ]:
run("""
""")

Which ones are right, and **why** is the odd one out wrong?

Then count the artists who are **not** active, using
`WHERE still_active != 1`. Add the two answers together. Is that the
number of artists? If not, where did the missing one go?

In [ ]:
run("""
""")

*Your answer here.*

## Stretch 2: tracks played on every device

Which tracks have been played on all five devices? Show the track, the
number of devices and the number of plays, most played first.

In [ ]:
run("""
""")

Now the same, but only counting plays that were **not skipped**. A
track that was only ever skipped in the car should not count as played
in the car.

Where does `skipped = 0` go: in the `WHERE` or in the `HAVING`? Try
both. Do they give the same number of rows? Which one is right, and why?

In [ ]:
run("""
""")

In [ ]:
run("""
""")

*Your answer here.*

## Stretch 3: do I listen more at the weekend?

Two new pieces for this one.

**`strftime('%w', played_at)`** gives the day of the week, as text:
`'0'` for Sunday, `'1'` for Monday, up to `'6'` for Saturday. (The same
function with `'%Y'` gives the year, which homework question 5 needs.)

**`CASE`** is SQL's `if`. It makes a new value for each row:

```sql
CASE
    WHEN minutes_played < 1 THEN 'short'
    ELSE 'long'
END AS length
```

Try both in the cell below first.

In [ ]:
run("""
SELECT played_at,
       strftime('%w', played_at) AS weekday,
       minutes_played,
       CASE
           WHEN minutes_played < 1 THEN 'short'
           ELSE 'long'
       END AS length
FROM plays
LIMIT 5
""")

Now write a query with one row for `'weekend'` and one for `'weekday'`,
and the number of plays in each. You can `GROUP BY` the name you gave
your `CASE`.

Then look at the answer and ask: **is that a fair comparison?** If not,
add a column that makes it fair. (`COUNT(DISTINCT played_at)` counts
days.)

In [ ]:
run("""
""")

*Your answer here.*

## Stretch 4: who did I not play in 2025?

Every artist with their number of plays **in 2025**, including the ones
with 0, fewest first.

Write the obvious version first: a `LEFT JOIN` and a `WHERE` for the year.
Before you run it, how many rows should come back? Then run it and look at
the row count.

In [ ]:
run("""
""")

If the count is not what you expected, the fix is to move the date
condition out of the `WHERE` and into the `ON`, with an `AND`. Try it, and
check the count again.

In [ ]:
run("""
""")

Now list **only** the artists with no 2025 plays at all. Keep the same
join, and add `WHERE p.play_id IS NULL`.

In [ ]:
run("""
""")

**Why** did the first `WHERE` break the left join, when this `WHERE` does
not? Both run after the join. Think about which rows each one keeps.

*Your answer here.*

## Stretch 5: spot the bug

Someone wants the number of awards and the number of plays for each genre,
and writes this. Before you run it, use what you know: there are 33 awards
in the whole table. What should the awards column add up to?

In [ ]:
run("""
SELECT a.genre,
       COUNT(w.award_id) AS awards,
       COUNT(p.play_id)  AS plays
FROM artists a
LEFT JOIN awards w USING (artist_id)
LEFT JOIN plays  p USING (artist_id)
GROUP BY a.genre
ORDER BY a.genre
""")

1. What went wrong, and why? Fjord & Flint (Folk, 3 awards, 237 plays) is
   a good artist to work it out with.
2. Some genres have the right play count anyway. Which ones, and why
   them?

*Your answer here.*

Now fix it by changing only the two `COUNT`s, using `DISTINCT`.

In [ ]:
run("""
""")

In notebook 03, `SUM(DISTINCT minutes_played)` was a trap that gave the
wrong answer. **Why is `COUNT(DISTINCT w.award_id)` a correct fix here?**
What is different about `award_id` compared with `minutes_played`?

*Your answer here.*

**Harder:** add total minutes per genre as a third column. `DISTINCT` will
not save you this time, so you will need CTEs.

In [ ]:
run("""
""")

## Stretch 6: 2024 against 2025

For every artist: minutes in 2024, minutes in 2025, and the change. Biggest
riser at the top.

Build it in steps, with a CTE for each:

1. `minutes_2024`: one row per artist, their 2024 minutes
2. `minutes_2025`: the same for 2025
3. `compared`: every artist, with both numbers side by side

CTEs are separated by commas, and a later one can use the earlier ones.

Two things to watch:

* Some artists have minutes in 2024 and none in 2025. Make sure they do not
  disappear. Which table should `compared` start from?
* What is `NULL` minus a number? `COALESCE` from the live exercise will
  help.

In [ ]:
run("""
""", limit=40)

**Harder:** which artists did I stop listening to? Played in 2024, not at
all in 2025. With `compared` already built, this should be a small change
at the bottom.

In [ ]:
run("""
""")

## Stretch 7: the same fix, three ways

In the session the fan-out fix was a CTE. There are two other ways to write
the same idea, and you will meet both in other people's SQL.

**A nested subquery.** Take the CTE from notebook 03 and move the
`per_artist` query into the `FROM`, in brackets, with `AS s` after it. You
should get the same three rows: Glass Tram, DJ Kompas, Fjord & Flint.

In [ ]:
run("""
""")

**A subquery after `IN`.** How many plays are by Dutch artists? Answer it
without a join: `WHERE artist_id IN (SELECT ... )`. Then answer it again with
a join, and check that both agree.

In [ ]:
run("""
""")

In [ ]:
run("""
""")

**A view.** A view is a saved query that behaves like a table. It stores the
query, not the data, so it is always current.

`CREATE TEMP VIEW` makes one that lasts only as long as this connection, so
nothing gets written into `music.db`. Run this:

In [ ]:
con.execute("DROP VIEW IF EXISTS artist_summary")
con.execute("""
CREATE TEMP VIEW artist_summary AS
SELECT a.artist_id,
       a.artist_name,
       a.country,
       a.genre,
       COUNT(p.play_id)                AS plays,
       ROUND(SUM(p.minutes_played), 1) AS minutes
FROM artists a
LEFT JOIN plays p USING (artist_id)
GROUP BY a.artist_id
""")
print("view created")

Now use `artist_summary` like any table. Which artists have 0 plays? And
which three countries have the most minutes? Compare that with question 1
of the joins exercise.

In [ ]:
run("""
""")

In [ ]:
run("""
""")

**Which of the three would you use, and when?** Nested subquery, CTE, or
view?

*Your answer here.*

## Stretch 8: the top artist in every genre

Who do I listen to most (by minutes) in each genre? One row per genre.

`GROUP BY` cannot answer this. It gives you one number per genre, not the
name of who won. You need something new.

**New tool: window functions.** A window function calculates something
across a set of rows *without* squashing them into one row the way
`GROUP BY` does. Every row stays, and gets a new column. Run this:

In [ ]:
run("""
WITH per_genre AS (
    SELECT a.genre, COUNT(*) AS plays
    FROM plays p
    JOIN artists a USING (artist_id)
    GROUP BY a.genre
)
SELECT genre, plays,
       RANK() OVER (ORDER BY plays DESC) AS position
FROM per_genre
""")

`RANK() OVER (ORDER BY plays DESC)` numbers the rows 1, 2, 3, in that
order. Add `PARTITION BY` and the numbering restarts for each group:

```sql
RANK() OVER (PARTITION BY a.genre ORDER BY s.minutes DESC)
```

Now the question. Build it in steps:

1. a CTE with minutes per artist (aggregate first, as always)
2. a CTE that joins on `artists` and ranks each artist within their genre
3. a final `SELECT` that keeps only position 1

In [ ]:
run("""
""")

**Why** does the "keep only position 1" filter need its own step? Try
putting `WHERE position = 1` in the same `SELECT` as the `RANK()` and read
the error. The execution order slide explains it.

*Your answer here.*

**Harder:** the top two per genre.

In [ ]:
run("""
""", limit=16)

In [ ]:
con.close()